# Notebook 11 — ArUco Marker Theory
### 6D Pose Vision Workshop · Part 5: ArUco Markers

**Estimated time:** 40 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video note 1 (*ArUco Marker Pose Estimation*), video note 12 (*How to Detect ArUco Markers*)

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV: {cv2.__version__}")

# Verify ArUco module is available (requires opencv-contrib-python)
try:
    _ = cv2.aruco.DICT_4X4_50
    print("ArUco module: available")
except AttributeError:
    print("ERROR: ArUco not available. Install: pip install opencv-contrib-python")

def show_images(pairs, figsize_per=(5, 5)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

## Learning Objectives

By the end of this notebook you will:

- Know what ArUco markers are and where the name comes from
- Understand the **binary grid encoding** and how IDs are embedded in the pattern
- Know what **Hamming distance** is and why it prevents false detections
- Understand **ArUco dictionaries** and how to choose the right one
- Know the tradeoff between **marker size** and **detection range**
- Understand why ArUco is the most practical choice for robotics infrastructure

---

## 1. What Are ArUco Markers?

> "ArUco" stands for **Augmented Reality University of Cordoba** — it was developed at the University of Córdoba, Spain. — Video Note 1

ArUco markers are **square binary patterns** — black and white grids that encode a unique ID in their pixel pattern. They are designed to be:

- **Detected fast** — corner detection is O(n) in image pixels
- **Robust** — work under partial occlusion, motion blur, steep angles
- **Unique** — each ID has a pattern far enough from other IDs to avoid confusion
- **Pose-able** — 4 corners give exact 6D pose via solvePnP

### Why ArUco beats chessboards for robotics

| Feature | Chessboard | ArUco |
|---|---|---|
| Size on wall/station | Large (30+ cm) | Small (5–20 cm) |
| Detection time | Slow (~50ms for 9x6) | Fast (<5ms) |
| Multiple simultaneous | 1 at a time | Many at once |
| Identity encoding | No — all look the same | Yes — each has unique ID |
| Partial occlusion | Fails | Works (if >3 corners visible) |
| Print and forget | No — must calibrate size | Yes — just know the side length |

---

## 2. The Binary Grid — How IDs Are Encoded

### Structure of a marker

An ArUco marker consists of:

1. **Black border** — 1-cell wide frame; enables detection (high contrast against white background)
2. **Data cells** — the inner grid; encodes the ID as a binary pattern

```
A 4×4 ArUco marker (DICT_4X4 family):

 ┌─────────────────────┐
 │ ■ ■ ■ ■ ■ ■ ■ ■ ■ │  ← black border (outer ring)
 │ ■ □ □ ■ □ □ ■ □ ■ │
 │ ■ □ □ □ □ □ □ □ ■ │  ← inner 4×4 data cells
 │ ■ ■ □ □ ■ □ □ □ ■ │    (■=black=0, □=white=1)
 │ ■ □ □ ■ □ □ □ ■ ■ │
 │ ■ ■ □ ■ □ ■ □ □ ■ │
 │ ■ ■ ■ ■ ■ ■ ■ ■ ■ │  ← black border
 └─────────────────────┘
   6×6 total cells (4+2 border)
```

### Reading the binary code

The inner data cells form a binary number:

- **Black cell** = 0
- **White cell** = 1

For a 4×4 marker: 16 data bits → theoretically $2^{16} = 65,536$ possible patterns.

But we use only a small subset — selected so that no two IDs look too similar.

### Rotation invariance

The binary code is designed so that you can determine the marker's orientation just by reading the bits — the pattern looks different when rotated 90°, 180°, 270°. This is how detection recovers which corner is which.

---

In [ ]:
# Generate and display several ArUco markers to visualize the binary patterns

os.makedirs('../assets/aruco_markers', exist_ok=True)

DICT_NAME = cv2.aruco.DICT_4X4_50
aruco_dict = cv2.aruco.getPredefinedDictionary(DICT_NAME)

MARKER_PX = 200  # size in pixels for generation

markers = []
for marker_id in range(6):
    # Generate marker image
    marker_img = cv2.aruco.generateImageMarker(aruco_dict, marker_id, MARKER_PX)
    
    # Save to file
    save_path = f'../assets/aruco_markers/4x4_id{marker_id:03d}.png'
    cv2.imwrite(save_path, marker_img)
    
    markers.append((marker_img, f'4x4_50 ID={marker_id}'))

show_images(markers[:6], figsize_per=(3, 3))

print(f"All markers are {MARKER_PX}×{MARKER_PX} pixels")
print(f"Each has a unique binary pattern in the 4×4 inner grid")
print(f"Notice: all look different — that's the point!")

In [ ]:
# Decode and visualize the binary bits in a marker

def show_marker_bits(marker_img, grid_size=4):
    """
    Visualize the binary encoding of an ArUco marker.
    Extracts the inner grid and shows each cell's value.
    """
    h, w = marker_img.shape
    
    # The marker has a 1-cell border, so inner grid starts at cell 1
    total_cells = grid_size + 2  # +2 for border
    cell_size = h // total_cells
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Left: the marker itself
    axes[0].imshow(marker_img, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'ArUco marker ({grid_size}×{grid_size} inner grid)')
    axes[0].axis('off')
    
    # Right: annotated with bit values
    display = cv2.cvtColor(marker_img, cv2.COLOR_GRAY2BGR)
    bit_grid = np.zeros((grid_size, grid_size), dtype=int)
    
    for row in range(grid_size):
        for col in range(grid_size):
            # Sample center of each inner cell
            cy = int((row + 1 + 0.5) * cell_size)
            cx = int((col + 1 + 0.5) * cell_size)
            
            # Sample 3x3 neighborhood
            patch = marker_img[cy-5:cy+5, cx-5:cx+5]
            bit = 1 if np.mean(patch) > 128 else 0
            bit_grid[row, col] = bit
            
            color = (0, 200, 0) if bit == 1 else (200, 0, 0)
            cv2.putText(display, str(bit), (cx-8, cy+8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    axes[1].imshow(display[:,:,::-1])
    axes[1].set_title(f'Binary bits (green=1=white, red=0=black)\n{bit_grid.flatten().tolist()}')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    
    return bit_grid

# Show bits for marker ID 0
marker_id0 = cv2.aruco.generateImageMarker(aruco_dict, 0, 300)
bits_id0 = show_marker_bits(marker_id0, grid_size=4)
print(f"\nBit grid for ID 0:")
print(bits_id0)
print(f"\nFlattened binary: {bits_id0.flatten().tolist()}")
print(f"As integer: {int(''.join(str(b) for b in bits_id0.flatten()), 2)} (approximate — actual encoding uses Hamming codes)")

## 3. Hamming Distance — Why Markers Don't Get Confused

### The problem

If two marker IDs have very similar bit patterns, a slightly blurry or partially occluded marker might be misidentified. In robotics, misidentifying a docking station marker could cause a collision.

### The solution: Hamming distance

The **Hamming distance** between two binary strings is the number of positions where they differ:

```
Pattern A: 1 0 1 1 0 0 1 0
Pattern B: 1 1 1 0 0 0 1 0
              ↑   ↑
             diff diff

Hamming distance(A, B) = 2   (2 positions differ)
```

ArUco dictionaries are designed so that the **minimum Hamming distance** between any two marker IDs is large:

$$d_{\text{Hamming}}(\text{ID}_i, \text{ID}_j) \geq d_{\text{min}} \quad \forall i \neq j$$

A larger minimum Hamming distance means:
- The system can **correct** more bit errors (up to $\lfloor (d_{\text{min}}-1)/2 \rfloor$ errors)
- False positives require **many** bits to flip simultaneously (very unlikely)

### Why not use all possible patterns?

A 4×4 grid has 16 bits → 65,536 possible patterns. But if we use all of them:
- Some patterns would have Hamming distance 1 from others
- A single noisy pixel → wrong ID
- False positives become common

ArUco dictionaries include only patterns with sufficient separation. The `4X4_50` dictionary has 50 markers, all with minimum Hamming distance of 12 from each other.

---

In [ ]:
# Demonstrate Hamming distance

def hamming_distance(a, b):
    """Compute Hamming distance between two binary arrays."""
    return int(np.sum(a != b))

# Example with simple bit patterns
examples = [
    ([1,0,1,1,0,0,1,0], [1,1,1,0,0,0,1,0], "2 bits differ"),
    ([1,0,0,0,0,0,0,0], [0,0,0,0,0,0,0,1], "2 bits differ"),
    ([1,0,1,0,1,0,1,0], [0,1,0,1,0,1,0,1], "8 bits differ (complement)"),
]

print("Hamming distance examples:")
for a, b, desc in examples:
    a_arr, b_arr = np.array(a), np.array(b)
    d = hamming_distance(a_arr, b_arr)
    print(f"  A: {a}")
    print(f"  B: {b}")
    print(f"  → Hamming distance = {d}  ({desc})\n")

# Check Hamming distances between actual ArUco marker patterns
# We'll look at the dictionary's byte codes
dict_4x4_50 = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
dict_4x4_250 = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_250)

print("ArUco dictionaries and their properties:")
print(f"  DICT_4X4_50:  50 markers  (for small robotics applications)")
print(f"  DICT_4X4_250: 250 markers (for larger deployments)")
print(f"  Fewer markers → larger min Hamming distance → more robust")

## 4. ArUco Dictionaries — Choosing the Right One

### Available dictionaries

| Dictionary | Grid size | # Markers | Min Hamming | Use case |
|---|---|---|---|---|
| `DICT_4X4_50` | 4×4 | 50 | 12 | Small deployments, high robustness |
| `DICT_4X4_100` | 4×4 | 100 | 12 | Medium |
| `DICT_4X4_250` | 4×4 | 250 | 10 | Large deployments |
| `DICT_4X4_1000` | 4×4 | 1000 | 8 | Very large, less robust |
| `DICT_5X5_50` | 5×5 | 50 | 22 | High robustness, bigger markers needed |
| `DICT_5X5_100` | 5×5 | 100 | 20 | Good balance |
| `DICT_6X6_250` | 6×6 | 250 | 34 | Best robustness, larger markers |
| `DICT_7X7_250` | 7×7 | 250 | 48 | Maximum robustness |
| `DICT_APRILTAG_36h11` | AprilTag | 587 | 9 | AprilTag family |

### How to choose

From video note 1:

| Project scale | Recommendation |
|---|---|
| 1–10 markers (single robot, 2–3 stations) | `DICT_4X4_50` — fastest, most robust |
| 10–100 markers (small warehouse) | `DICT_5X5_100` |
| 100–500 markers (large facility) | `DICT_6X6_250` |
| 500+ markers (campus, multi-building) | `DICT_7X7_1000` |

**Rule of thumb:** Use the smallest dictionary that has enough IDs for your application. Fewer IDs = more Hamming separation = more robust detection.

### Grid size vs detection range

Larger grid sizes (6×6, 7×7) require more pixels per marker for reliable detection. A 4×4 marker is detectable at ~60 pixels across; a 7×7 needs ~100+ pixels.

```
Detection range at 640×480 with fx=600:

Marker side = 20cm, DICT_4X4 (needs ~60px across):
  min_px = 60, fx = 600, marker_m = 0.20
  max_Z ≈ fx × marker_m / min_px = 600 × 0.20 / 60 = 2.0 m

Same marker, DICT_7X7 (needs ~100px across):
  max_Z ≈ 600 × 0.20 / 100 = 1.2 m   (shorter range!)
```

In [ ]:
# Compare markers from different dictionaries (same ID, different size/complexity)

dicts_to_compare = [
    (cv2.aruco.DICT_4X4_50,  '4x4_50\n(4×4 grid)'),
    (cv2.aruco.DICT_5X5_50,  '5x5_50\n(5×5 grid)'),
    (cv2.aruco.DICT_6X6_50,  '6x6_50\n(6×6 grid)'),
    (cv2.aruco.DICT_7X7_50,  '7x7_50\n(7×7 grid)'),
]

marker_id = 7  # same ID in all dicts
comparison = []
for dict_code, name in dicts_to_compare:
    d = cv2.aruco.getPredefinedDictionary(dict_code)
    m = cv2.aruco.generateImageMarker(d, marker_id, 200)
    comparison.append((m, f'{name}\nID={marker_id}'))

show_images(comparison, figsize_per=(3, 3))
print("Same ID (7) in different dictionaries — more complex = more bits = harder to detect at distance")

In [ ]:
# Compute detection range for different configurations

def estimate_detection_range(marker_size_m, fx_px, min_pixels_needed, img_w=640, img_h=480):
    """
    Estimate max detection range using pinhole projection:
    Z = fx * marker_size / pixel_width
    """
    max_Z = fx_px * marker_size_m / min_pixels_needed
    return max_Z

configs = [
    # (marker_m, fx, min_px, label)
    (0.05, 600, 60,  '5cm marker, DICT_4X4'),
    (0.10, 600, 60,  '10cm marker, DICT_4X4'),
    (0.20, 600, 60,  '20cm marker, DICT_4X4'),
    (0.20, 600, 100, '20cm marker, DICT_7X7'),
    (0.30, 800, 60,  '30cm marker, DICT_4X4 (better camera)'),
]

print("Estimated maximum detection range:")
print(f"{'Configuration':<40} {'Max range (m)'}")
print("-" * 55)
for marker_m, fx, min_px, label in configs:
    max_Z = estimate_detection_range(marker_m, fx, min_px)
    print(f"{label:<40} {max_Z:.2f} m")

print("\nPractical rule:")
print("  Larger marker → detect from farther away")
print("  Simpler dictionary (4x4) → detect at shorter pixel width → longer range")
print("  Better camera (higher fx) → detect at longer range")

## 5. Why ArUco Is Robust for Robotics

### From video note 1 (observations from demo):

The detection system showed:
- **Very good real-time performance** — processes at 30+ FPS
- **Fast motion tracking** — handles quick camera movement
- **Detection under glare** — works with some specular reflection
- **Detection under partial occlusion** — 3 corners visible = still detectable
- **Detection at steep angles** — up to ~70° off-normal

### Practical limitations

| Scenario | Impact |
|---|---|
| >50% occluded | Detection fails |
| >75° angle | Detection unreliable |
| Heavy motion blur | Detection fails |
| Very small in image (<50px) | ID reading fails (corners found but bits unclear) |
| Reflective or glossy surface | Partial failure |

### Printing tips for reliable detection

1. **Matte paper** — glossy paper causes glare that obscures cells
2. **High contrast** — printer ink should be dark black, not gray
3. **Rigid backing** — mount on foam board or thin aluminium; paper warps and reduces pose accuracy
4. **White border** — print with extra white margin (1 marker-width) around the black border
5. **Measure the actual printed size** — printers scale; measure with calipers and use that exact value in `estimatePoseSingleMarkers`

In [ ]:
# Generate a complete set of markers for a robotics scenario
# 3 markers for: charging station, conveyor station A, conveyor station B

DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)

stations = [
    (0, 'Charging Station'),
    (1, 'Conveyor Station A'),
    (2, 'Conveyor Station B'),
]

marker_imgs = []
for marker_id, name in stations:
    # Generate at 400px (print at 10cm → 1 cm per square)
    m = cv2.aruco.generateImageMarker(DICT, marker_id, 400)
    
    # Add white border and label for printing
    border = 40
    h, w = m.shape
    labeled = np.ones((h + border*2 + 40, w + border*2), dtype=np.uint8) * 255
    labeled[border:border+h, border:border+w] = m
    
    # Add station label
    cv2.putText(labeled, f'ID={marker_id}: {name}',
                (border, h + border + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, 0, 2)
    
    save_path = f'../assets/aruco_markers/station_{marker_id:02d}_{name.lower().replace(" ","_")}.png'
    cv2.imwrite(save_path, labeled)
    marker_imgs.append((labeled, f'ID={marker_id}\n{name}'))
    print(f"Saved: {save_path}")

show_images(marker_imgs, figsize_per=(4, 5))
print("\nReady to print! Mount on rigid backing at each station.")
print("Measure the actual inner marker size (in meters) before using for pose estimation.")

## Recap

| Concept | Key point |
|---|---|
| ArUco name | Augmented Reality University of Córdoba |
| Structure | Black border + inner NxN data grid |
| Bit encoding | Black cell=0, White cell=1; pattern encodes unique ID |
| Hamming distance | Min bits that differ between any two IDs; larger = more robust |
| Dictionary | Set of markers with guaranteed minimum Hamming separation |
| Choosing size | Fewer markers → more robust; more markers → more scalable |
| Detection | 4X4 needs ~60px wide; 7X7 needs ~100px wide — affects max range |
| Why for robotics | Fast, unique IDs, 4 corners → 6D pose, partial-occlusion robust |
| Printing | Matte paper, rigid backing, measure actual printed size |

---

**Next:** `12_generate_aruco.ipynb` — generate, print, and organize ArUco markers for your specific robotics project.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Detection range calculator
# ============================================================
# Write a function plan_aruco_deployment(camera_K, img_size, max_range_m)
# that recommends:
#   - Minimum marker size to use at max_range_m with reliable detection
#   - Best dictionary to use for robustness
#
# Use the formula: marker_size = max_Z * min_pixels / fx
# Assume min_pixels = 80 for reliable detection
#
# Test with:
#   K = np.array([[800,0,320],[0,800,240],[0,0,1]])
#   img_size = (640, 480)
#   max_range = 3.0 m

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def plan_aruco_deployment(K, img_size, max_range_m, min_pixels=80):
#     fx = K[0, 0]
#     min_marker_m = max_range_m * min_pixels / fx
#
#     print(f"Camera: fx={fx:.0f}px, image {img_size[0]}x{img_size[1]}")
#     print(f"Required detection range: {max_range_m:.1f} m")
#     print(f"\nMinimum marker size: {min_marker_m*100:.1f} cm")
#
#     # Recommend dict based on number of stations
#     print("\nDictionary recommendations:")
#     for n_markers, dict_name, notes in [
#         (50,   'DICT_4X4_50',  'best robustness, up to 50 stations'),
#         (250,  'DICT_5X5_250', 'up to 250 stations, slightly less robust'),
#         (1000, 'DICT_6X6_1000','large deployment, 1000+ stations'),
#     ]:
#         print(f"  {dict_name}: {notes}")
#
# K_test = np.array([[800,0,320],[0,800,240],[0,0,1]], dtype=np.float64)
# plan_aruco_deployment(K_test, (640,480), max_range_m=3.0)

In [ ]:
# ============================================================
# EXERCISE 2: Dictionary exploration
# ============================================================
# Generate marker ID=5 from 4 different dictionaries:
#   DICT_4X4_50, DICT_5X5_50, DICT_6X6_50, DICT_ARUCO_ORIGINAL
# Display them side by side.
# Then answer: why does the 4x4 version have the fewest white cells?

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# dicts = [
#     (cv2.aruco.DICT_4X4_50,         'DICT_4X4_50'),
#     (cv2.aruco.DICT_5X5_50,         'DICT_5X5_50'),
#     (cv2.aruco.DICT_6X6_50,         'DICT_6X6_50'),
#     (cv2.aruco.DICT_ARUCO_ORIGINAL, 'DICT_ARUCO_ORIGINAL'),
# ]
# pairs = []
# for code, name in dicts:
#     d = cv2.aruco.getPredefinedDictionary(code)
#     m = cv2.aruco.generateImageMarker(d, 5, 200)
#     pairs.append((m, f'{name}\nID=5'))
# show_images(pairs, figsize_per=(3,3))
#
# # Answer: The 4x4 uses only 16 data bits to encode all IDs.
# # The encoding prioritizes maximal Hamming distance — the specific bit
# # pattern for each ID is chosen to maximize separation from others,
# # not to look 'prettier'. Different dictionaries use completely
# # different encoding schemes (4x4 vs 5x5 have different grid sizes
# # AND different binary code designs).